In [ ]:
# =============================================================================
#@markdown # **CELL 1**: Git Clone / Pull + Instalar Dependências + Klein 9B (sd.cpp pré-compilado)
# =============================================================================
REPO_URL = "https://github.com/GuiAworld/GradioKlein.git" #@param {type:"string"}
APP_DIR = "/content/GradioKlein"
DEBUG_MODE = False #@param {type:"boolean"}

import os
import subprocess

if os.path.exists(os.path.join(APP_DIR, ".git")):
    print("🔄 Repositório já existe — executando git pull...")
    subprocess.run(["git", "-C", APP_DIR, "pull", "--ff-only"], check=True)
else:
    print(f"📥 Clonando {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, APP_DIR], check=True)

try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
    else:
        print("⚠️ HF_TOKEN não configurado nos Secrets (prosseguindo sem token).")
except Exception:
    print("⚠️ HF_TOKEN não encontrado nos Secrets (prosseguindo sem token).")

mode_flag = "--mode debug" if DEBUG_MODE else "--mode prod"
cmd = f"python {APP_DIR}/install.py {mode_flag} --models-dir /content/models --sdcpp-dir /content/sdcpp_bin"
print(f"⚙️ Executando: {cmd}")
get_ipython().system(cmd)
print("✅ Cell 1 concluída!")


In [ ]:
# =============================================================================
#@markdown # **CELL 2**: Download de LoRAs (Opcional — FLUX.2 Klein 9B)
# =============================================================================
import sys
sys.path.insert(0, "/content/GradioKlein")
from lora_downloader import download_hf_loras, download_url_loras

LORA_DIR = "/content/LoRAs"

# Cole aqui links diretos do Civitai / URLs compatíveis com FLUX.2 Klein 9B
# (Obs: LoRAs de Klein 4B não são compatíveis com a arquitetura Klein 9B)
DOWNLOAD_LINKS_9B = [
]

# Lista de arquivos opcionais do HuggingFace: (repo_id, ["arquivo.safetensors"])
HF_LORA_FILES = []

try:
    from google.colab import userdata
    civitai_token = userdata.get("CIVITAI_API_TOKEN")
except Exception:
    civitai_token = None

for repo_id, files in HF_LORA_FILES:
    download_hf_loras(LORA_DIR, repo_id, files)

if DOWNLOAD_LINKS_9B:
    download_url_loras(LORA_DIR, DOWNLOAD_LINKS_9B, civitai_token=civitai_token)
else:
    import os
    os.makedirs(LORA_DIR, exist_ok=True)
    print(f"ℹ️ Nenhuma LoRA configurada para download em {LORA_DIR} (pulando).")


In [ ]:
# =============================================================================
#@markdown # **CELL 3**: Iniciar `sd-server` (9B GGUF) + App Gradio + Túnel
# =============================================================================
DEBUG_MODE = False #@param {type:"boolean"}
USE_CLOUDFLARE = True #@param {type:"boolean"}
USE_GRADIO_SHARE = True #@param {type:"boolean"}
CF_TUNNEL_TOKEN = "" #@param {type:"string"}

import os

token_to_use = CF_TUNNEL_TOKEN.strip()
if not token_to_use:
    try:
        from google.colab import userdata
        token_to_use = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        token_to_use = ""

flags = [
    "--models-dir /content/models",
    "--sdcpp-dir /content/sdcpp_bin",
    "--lora-dir /content/LoRAs",
    "--host 0.0.0.0",
    "--port 7860",
]
if DEBUG_MODE:
    flags.append("--debug")
if USE_CLOUDFLARE:
    flags.append("--cloudflare")
if USE_GRADIO_SHARE:
    flags.append("--share")
if token_to_use:
    os.environ["CF_TUNNEL_TOKEN"] = token_to_use

launch_cmd = "python /content/GradioKlein/app.py " + " ".join(flags)
print(f"🚀 Iniciando: {launch_cmd}")
get_ipython().system(launch_cmd)
